# Example Workflow

Demonstrates the full library with synthetic data. No real data needed.

**Contents:**
1. Generate synthetic animals
2. Data class hierarchy and inspection
3. Filtering pipeline
4. Low-level functions (raw arrays)
5. Session-level analysis (compute_ functions)
6. Plotting (plot_ functions)
7. Comparing two conditions
8. Data class convenience methods (thin wrappers)
9. Learning trajectory with per-session parameters
10. Custom summary statistics


## Setup


In [ ]:
from pathlib import Path
import os, sys

_NOTEBOOK_DIR = Path(os.path.abspath(''))
_BEHAV_UTIL_DIR= _NOTEBOOK_DIR.parent
_PROJECT_ROOT = _BEHAV_UTIL_DIR.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from behav_utils import (
    # Data structures
    ExperimentData, AnimalData, SessionData,
    # Synthetic data
    generate_synthetic_animal, sample_stimuli,
    # Session selection
    select_sessions, list_presets,
    # Filtering
    filter_trials, filter_session, pool_arrays, build_mask, opto_mask,
    # Low-level analysis (arrays)
    fit_psychometric, compute_update_matrix, compare_conditions,
    cumulative_gaussian, compute_summary_stats, list_available_stats,
    # Session-level analysis (sessions -> result dicts)
    compute_psychometric, compute_um, compute_trajectory,
    compute_comparison, compute_session_raster,
    # Plotting (result dicts -> axes)
    plot_psychometric, plot_um, plot_trajectory,
    plot_comparison, plot_session_raster,
    # Styles
    PALETTE, COLOURS, apply_style,
)
from behav_utils.data.synthetic import noisy_psychometric_simulator
from behav_utils.analysis.session_features import build_feature_matrix

apply_style()
print('All imports successful.')


## 1. Generate synthetic data

`generate_synthetic_animal` creates a full `AnimalData` with `SessionData` objects.
The simulator callable has signature `(stimuli, categories, rng, **kwargs) -> choices`.
Parameters go in `simulator_kwargs`.


In [ ]:
# Two animals with different psychometric parameters
animal_a, info_a = generate_synthetic_animal(
    animal_id='SynA',
    n_sessions=12,
    trials_per_session=150,
    simulator=noisy_psychometric_simulator,
    simulator_kwargs={'sigma': 0.15, 'lapse': 0.02},
    seed=42,
)

animal_b, info_b = generate_synthetic_animal(
    animal_id='SynB',
    n_sessions=12,
    trials_per_session=150,
    simulator=noisy_psychometric_simulator,
    simulator_kwargs={'sigma': 0.40, 'lapse': 0.10},
    seed=99,
)

print(f'Animal A: {animal_a.n_sessions} sessions, sigma=0.15, lapse=0.02')
print(f'Animal B: {animal_b.n_sessions} sessions, sigma=0.40, lapse=0.10')
print(f'Generation info: {info_a}')


## 2. Data class hierarchy

```
ExperimentData
  └── AnimalData
        └── SessionData
              ├── SessionMetadata
              └── TrialData
```


In [ ]:
# Build an ExperimentData from our two animals
experiment = ExperimentData()
experiment.add_animal(animal_a)
experiment.add_animal(animal_b)

print(f'Experiment: {experiment.n_animals} animals')
print(f'Animal IDs: {list(experiment.animals.keys())}')


In [ ]:
# AnimalData
animal = experiment.get_animal('SynA')
print(f'Animal: {animal.animal_id}')
print(f'Sessions: {animal.n_sessions}')
print(f'Session IDs: {animal.session_ids[:5]}...')

In [ ]:
# SessionData
session = animal.sessions[5]
print(f'Session: {session.session_id}')
print(f'Date: {session.date}')
print(f'Trials: {session.n_trials}')
print(f'Stage: {session.stage}')
print(f'Distribution: {session.distribution}')


In [ ]:
# TrialData — per-trial arrays
trials = session.trials
print(f'Stimulus range: [{trials.stimulus.min():.2f}, {trials.stimulus.max():.2f}]')
print(f'Choice values: {np.unique(trials.choice[~np.isnan(trials.choice)])}')
print(f'Abort rate: {trials.abort.mean():.2%}')
print(f'No-response rate: {trials.no_response.mean():.2%}')


In [ ]:
# SessionMetadata — constant within session
meta = session.metadata
print(f'Stage: {meta.stage}')
print(f'Animal ID: {meta.animal_id}')
print(f'All fields: {meta.fields}')


## 3. Filtering pipeline

All filtering lives in `filtering.py`. The pipeline is always explicit:
`select_sessions` → `filter_trials` → analysis/plotting.


In [ ]:
# Available presets
print('Session filter presets:')
for name in sorted(list_presets()):
    print(f'  {name}')


In [ ]:
# Select sessions (session-level filtering)
# Synthetic data has stage='Full_Task_Cont' and distribution='uniform' by default
all_sessions = list(animal_a.sessions)
print(f'All sessions: {len(all_sessions)}')

# Filter trials (trial-level filtering)
# filter_trials excludes aborts and opto trials by default
clean = filter_trials(all_sessions)
print(f'After filter_trials: {len(clean)} sessions')
print(f'First session: {clean[0].n_trials} trials (aborts removed)')


In [ ]:
# Custom filtering with mask functions
# Example: only trials where stimulus > 0 (category B stimuli)
from behav_utils.data.filtering import filter_trials

high_stim = filter_trials(
    all_sessions,
    mask_fn=lambda s: s.trials.stimulus > 0,
    label='high_stimulus',
)
print(f'High-stimulus sessions: {len(high_stim)}')
print(f'First session trials: {high_stim[0].n_trials}')
print(f'Filter info: {high_stim[0].filter_info}')


In [ ]:
# Pool arrays across sessions
pooled = pool_arrays(clean)
print(f'Pooled array keys: {sorted(pooled.keys())}')
print(f'Total trials pooled: {len(pooled["stimuli"])}')


## 4. Low-level functions (raw arrays)

These take numpy arrays directly — usable with model output, synthetic data,
or any custom source. No SessionData required.


In [ ]:
# Generate raw arrays
rng = np.random.default_rng(42)
stimuli, categories = sample_stimuli(500, distribution='uniform', rng=rng)
choices = noisy_psychometric_simulator(stimuli, categories, rng, sigma=0.20, lapse=0.03)

# fit_psychometric — core psychometric fitting
params = fit_psychometric(stimuli, choices)
print('fit_psychometric:')
for k in ('mu', 'sigma', 'lapse_low', 'lapse_high', 'success'):
    v = params.get(k)
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')


In [ ]:
# compute_update_matrix — stimulus-choice transition matrix
um, conditional, info = compute_update_matrix(stimuli, choices, categories)
print(f'UM shape: {um.shape}')
print(f'Trials used: {info["total_trials"]}')


In [ ]:
# compute_summary_stats — registered stats on raw arrays
stats = compute_summary_stats(
    choices, stimuli, categories,
    stat_names=['accuracy', 'recency', 'psychometric', 'win_stay', 'side_bias'],
    return_dict=True,
)
print('Summary stats:')
for k, v in stats.items():
    if isinstance(v, dict):
        for kk, vv in v.items():
            print(f'  {kk}: {vv:.4f}' if isinstance(vv, float) else f'  {kk}: {vv}')
    elif isinstance(v, float):
        print(f'  {k}: {v:.4f}')


In [ ]:
# compare_conditions — full two-condition comparison on raw arrays
rng2 = np.random.default_rng(99)
stim_2, cat_2 = sample_stimuli(500, distribution='uniform', rng=rng2)
ch_2 = noisy_psychometric_simulator(stim_2, cat_2, rng2, sigma=0.40, lapse=0.10)

comp = compare_conditions(
    stimuli, choices, categories,
    stim_2, ch_2, cat_2,
    n_permutations=500, n_bootstrap=500,
    label_a='Sharp', label_b='Shallow',
)
print(f'PSE diff: {comp["diffs"]["pse"]:.4f}')
print(f'Accuracy diff: {comp["diffs"]["accuracy"]:.4f}')
print(f'Permutation p (PSE): {comp["perm_p"]["pse"]:.4f}')
print(f'UM RMSE: {comp["um_rmse"]:.4f}')


In [ ]:
# list all available stats
print(f'{len(list_available_stats())} registered stats:')
print(', '.join(sorted(list_available_stats())))


## 5. Session-level analysis (compute_ functions)

These take pre-filtered `List[SessionData]` and return structured dicts.


In [ ]:
sessions_a = filter_trials(list(animal_a.sessions))
sessions_b = filter_trials(list(animal_b.sessions))

# compute_psychometric — three modes
psych_pooled = compute_psychometric(sessions_a, mode='pooled', n_bootstrap=200)
psych_overlay = compute_psychometric(sessions_a, mode='overlay')
psych_mean = compute_psychometric(sessions_a, mode='session_mean', n_bootstrap=100)

print(f'Pooled PSE: {psych_pooled["params"].get("mu", float("nan")):.4f}')
print(f'Overlay: {psych_overlay["n_sessions"]} session fits')
print(f'Session mean: {psych_mean["n_sessions"]} sessions')
print(f'\nResult keys (pooled): {sorted(psych_pooled.keys())}')


In [ ]:
# compute_um
um_a = compute_um(sessions_a)
um_b = compute_um(sessions_b)
print(f'UM keys: {sorted(um_a.keys())}')
print(f'UM shape: {um_a["um"].shape}, sessions: {um_a["n_sessions"]}')


In [ ]:
# compute_trajectory
traj_a = compute_trajectory(sessions_a, stat_names=['accuracy', 'pse', 'recency'])
traj_b = compute_trajectory(sessions_b, stat_names=['accuracy', 'pse', 'recency'])
print(f'Trajectory keys: {sorted(traj_a.keys())}')
print(f'Accuracy: {np.round(traj_a["values"]["accuracy"], 3)}')


In [ ]:
# compute_comparison (session-level)
comp_result = compute_comparison(
    sessions_a, sessions_b,
    n_permutations=500, n_bootstrap=500,
    label_a='Animal A', label_b='Animal B',
)
print(f'PSE diff: {comp_result["diffs"]["pse"]:.4f}')
print(f'Fisher p: {comp_result["fisher_p"]:.4f}')
print(f'Sessions: {comp_result["n_sessions_a"]}A vs {comp_result["n_sessions_b"]}B')


In [ ]:
# compute_session_raster
raster = compute_session_raster(sessions_a[0])
print(f'Raster keys: {sorted(raster.keys())}')
print(f'Trials: {raster["n_trials"]}, Correct: {raster["correct"].sum()}')


## 6. Plotting (plot_ functions)

Every plot function takes a result dict. No computation inside.
User controls layout with `plt.subplots()`. Overlay = same function, same axes, different colour.


In [ ]:
# Psychometric: two animals overlaid
fig, ax = plt.subplots(figsize=(5, 4))
psych_b_pooled = compute_psychometric(sessions_b, mode='pooled', n_bootstrap=200)
plot_psychometric(psych_pooled, ax=ax, color=PALETTE[0], label='Animal A')
plot_psychometric(psych_b_pooled, ax=ax, color=PALETTE[1], label='Animal B')
ax.legend(fontsize=9)
ax.set_title('Pooled Psychometric Comparison')
plt.tight_layout()
plt.show()


In [ ]:
# Psychometric: three modes side by side
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_psychometric(psych_pooled, ax=axes[0], color=PALETTE[0], title='Pooled')
plot_psychometric(psych_overlay, ax=axes[1], title='Overlay')
plot_psychometric(psych_mean, ax=axes[2], color=PALETTE[2], title='Session Mean')
plt.tight_layout()
plt.show()


In [ ]:
# Update matrices
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_um(um_a, ax=axes[0], title='Animal A')
plot_um(um_b, ax=axes[1], title='Animal B')
# Raw ndarray arithmetic then plot
plot_um(um_a['um'] - um_b['um'], ax=axes[2], title='A - B')
plt.tight_layout()
plt.show()


In [ ]:
# Trajectory: two animals, two stats
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for traj, colour, label in [(traj_a, PALETTE[0], 'A'), (traj_b, PALETTE[1], 'B')]:
    plot_trajectory(traj, 'accuracy', ax=axes[0], color=colour, label=label)
    plot_trajectory(traj, 'pse', ax=axes[1], color=colour, label=label)
axes[0].legend(fontsize=9); axes[0].set_title('Accuracy')
axes[1].legend(fontsize=9); axes[1].set_title('PSE')
plt.tight_layout()
plt.show()


In [ ]:
# Comparison plots
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_comparison(comp_result, ax=axes[0], metric='psychometric', title='Psychometric')
plot_comparison(comp_result, ax=axes[1], metric='accuracy', title='Accuracy')
plt.tight_layout()
plt.show()


In [ ]:
# Comparison: UM mode (creates 3-panel figure)
fig, axes = plot_comparison(comp_result, metric='um', title='UM Comparison')
plt.tight_layout()
plt.show()


In [ ]:
# Session raster
fig, ax = plot_session_raster(raster, window=20)
plt.tight_layout()
plt.show()


## 7. Comparing two conditions (full pattern)

The canonical comparison pattern:
1. Start from same sessions
2. Filter differently for each condition
3. Compute separately
4. Plot on same axes


In [ ]:
# Simulate an opto experiment by splitting trials randomly
# (real data would use session.trials.opto_mask)
sessions = filter_trials(list(animal_a.sessions))

# "Condition A": first half of trials per session
cond_a = filter_trials(
    list(animal_a.sessions),
    mask_fn=lambda s: np.arange(s.trials.n_trials) < s.trials.n_trials // 2,
    label='first_half',
)

# "Condition B": second half
cond_b = filter_trials(
    list(animal_a.sessions),
    mask_fn=lambda s: np.arange(s.trials.n_trials) >= s.trials.n_trials // 2,
    label='second_half',
)

# Compute and overlay
psych_a = compute_psychometric(cond_a, mode='pooled')
psych_b = compute_psychometric(cond_b, mode='pooled')

fig, ax = plt.subplots(figsize=(5, 4))
plot_psychometric(psych_a, ax=ax, color=PALETTE[0], label='First half')
plot_psychometric(psych_b, ax=ax, color=PALETTE[1], label='Second half')
ax.legend()
ax.set_title('Within-session split (should be similar)')
plt.tight_layout()
plt.show()


## 8. Data class convenience methods

Data classes have thin wrappers for quick exploration.
These call `compute_` then `plot_` internally.
For full control (colours, overlays, custom layouts), use the explicit pipeline.


In [ ]:
# AnimalData convenience methods
fig, ax = animal_a.plot_psychometric(mode='pooled')
plt.show()


In [ ]:
fig, ax = animal_a.plot_trajectory('accuracy')
plt.show()


In [ ]:
fig, ax = animal_a.plot_um()
plt.show()


In [ ]:
# SessionData convenience methods
session = sessions_a[5]

fig, ax = session.plot_psychometric()
plt.show()


In [ ]:
fig, ax = session.plot_trials(window=20)
plt.show()


In [ ]:
# SessionData.stats — summary statistics
stats = session.stats(['accuracy', 'pse', 'slope', 'recency', 'win_stay', 'side_bias'])
print(f'Stats for {session.session_id}:')
for k, v in stats.items():
    print(f'  {k:>15s}: {v:.4f}' if isinstance(v, float) else f'  {k:>15s}: {v}')


In [ ]:
# AnimalData.plot_overview — multi-panel summary
fig, axes = animal_a.plot_overview(stats=['accuracy', 'pse', 'recency'])
plt.show()


In [ ]:
# ExperimentData convenience methods
fig, ax = experiment.plot_trajectory('accuracy', combine='mean_sem')
plt.show()


In [ ]:
fig, ax = experiment.plot_psychometric(mode='pooled')
plt.show()


In [ ]:
# ExperimentData.summary
print(experiment.summary().to_string(index=False))


## 9. Feature matrix and session features


In [ ]:
# Per-session feature matrix for one animal
df = build_feature_matrix(animal_a)
print(f'Feature matrix: {df.shape[0]} sessions x {df.shape[1]} columns')
print(f'Columns: {list(df.columns[:10])}...')
print()
print(df[['session_idx', 'accuracy', 'pse', 'slope']].head())


## 10. Learning trajectory (per-session varying parameters)

`per_session_simulator_kwargs` lets you model changing parameters across sessions.


In [ ]:
n_sess = 25
animal_learn, _ = generate_synthetic_animal(
    animal_id='LEARN01',
    n_sessions=n_sess,
    trials_per_session=200,
    simulator=noisy_psychometric_simulator,
    per_session_simulator_kwargs=[
        {'sigma': 0.8 - i * 0.025, 'lapse': max(0.01, 0.15 - i * 0.005)}
        for i in range(n_sess)
    ],
    seed=456,
)

clean_learn = filter_trials(list(animal_learn.sessions))
traj_learn = compute_trajectory(clean_learn, ['accuracy', 'pse', 'slope'])

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for i, stat in enumerate(['accuracy', 'pse', 'slope']):
    plot_trajectory(traj_learn, stat, ax=axes[i], color=PALETTE[2])
    axes[i].set_title(stat)
plt.suptitle('Learning trajectory (sigma decreasing)', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Same thing via the data class wrapper
fig, axes = animal_learn.plot_overview(stats=['accuracy', 'pse', 'slope'])
plt.show()


## 11. Low-level functions with model predictions

Low-level functions accept raw arrays — use them for model-generated data,
posterior predictive checks, or theoretical curves.


In [ ]:
# Theoretical psychometric curves
x = np.linspace(-1, 1, 200)
y_sharp = cumulative_gaussian(x, mu=0.0, sigma=0.15, lapse_low=0.02, lapse_high=0.02)
y_shallow = cumulative_gaussian(x, mu=0.0, sigma=0.40, lapse_low=0.10, lapse_high=0.10)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(x, y_sharp, color=PALETTE[0], lw=2, label='Sharp (sigma=0.15)')
ax.plot(x, y_shallow, color=PALETTE[1], lw=2, label='Shallow (sigma=0.40)')
ax.axhline(0.5, ls='--', color='grey', alpha=0.3)
ax.axvline(0.0, ls='--', color='grey', alpha=0.3)
ax.set_xlabel('Stimulus'); ax.set_ylabel('P(choose B)')
ax.set_title('Model predictions (cumulative_gaussian)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# compute_psychometric also accepts raw (stimuli, choices) tuples
result_raw = compute_psychometric((stimuli, choices), mode='pooled')
print(f'PSE from raw tuple: {result_raw["params"].get("mu", float("nan")):.4f}')

fig, ax = plt.subplots(figsize=(5, 4))
plot_psychometric(result_raw, ax=ax, color=PALETTE[0], title='From raw arrays')
plt.tight_layout()
plt.show()


## 12. Custom summary statistics


In [ ]:
from behav_utils.analysis.summary_stats import register_stat

@register_stat('extreme_bias')
def extreme_bias(choices, stimuli, categories, **kwargs):
    """Fraction of trials where choice disagrees with stimulus sign."""
    valid = ~np.isnan(choices)
    if valid.sum() == 0:
        return np.nan
    stim_sign = (stimuli[valid] > 0).astype(float)
    return float(np.mean(choices[valid] != stim_sign))

# Now usable everywhere
stats = sessions_a[0].stats(['extreme_bias', 'accuracy'])
print(f'Extreme bias: {stats["extreme_bias"]:.4f}')
print(f'Accuracy: {stats["accuracy"]:.4f}')

# And in trajectories
traj = compute_trajectory(sessions_a, ['extreme_bias', 'accuracy'])
fig, ax = plt.subplots(figsize=(6, 3.5))
plot_trajectory(traj, 'extreme_bias', ax=ax, color=PALETTE[3], label='Extreme bias')
ax.set_title('Custom stat trajectory')
plt.tight_layout()
plt.show()


## Summary

### Three-level convention

| Level | Function | Input | Output |
|:------|:---------|:------|:-------|
| **Low-level** | `fit_psychometric(stim, ch)` | Raw arrays | Params dict |
| **Low-level** | `compute_update_matrix(s, ch, cat)` | Raw arrays | (um, cond, info) |
| **Low-level** | `compare_conditions(...)` | Raw arrays x 2 | Comparison dict |
| **Low-level** | `compute_summary_stats(ch, s, cat)` | Raw arrays | Stats dict |
| **Session-level** | `compute_psychometric(sessions)` | List[SessionData] | Result dict |
| **Session-level** | `compute_um(sessions)` | List[SessionData] | Result dict |
| **Session-level** | `compute_trajectory(sessions, stats)` | List[SessionData] | Result dict |
| **Session-level** | `compute_comparison(a, b)` | List x 2 | Result dict |
| **Session-level** | `compute_session_raster(session)` | SessionData | Result dict |
| **Plotting** | `plot_psychometric(result)` | Result dict | (fig, ax) |
| **Plotting** | `plot_um(result)` | Result dict | (fig, ax) |
| **Plotting** | `plot_trajectory(result, stat)` | Result dict | (fig, ax) |
| **Plotting** | `plot_comparison(result, metric)` | Result dict | (fig, ax) |
| **Plotting** | `plot_session_raster(result)` | Result dict | (fig, ax) |

### Data class convenience methods

| Class | Method | Wraps |
|:------|:-------|:------|
| `SessionData` | `.stats(stat_names)` | `compute_summary_stats` |
| `SessionData` | `.plot_psychometric()` | `compute_psychometric` + `plot_psychometric` |
| `SessionData` | `.plot_trials()` | `compute_session_raster` + `plot_session_raster` |
| `AnimalData` | `.plot_psychometric(mode=)` | `compute_psychometric` + `plot_psychometric` |
| `AnimalData` | `.plot_trajectory(stat)` | `compute_trajectory` + `plot_trajectory` |
| `AnimalData` | `.plot_um()` | `compute_um` + `plot_um` |
| `AnimalData` | `.plot_overview(stats=)` | Multiple compute + plot calls |
| `ExperimentData` | `.plot_trajectory(stat, combine=)` | Multi-animal compute + plot |
| `ExperimentData` | `.plot_psychometric(mode=)` | Pooled compute + plot |

### Pipeline pattern

```python
sessions = select_sessions(animal, preset='expert_uniform')
clean = filter_trials(sessions)
result = compute_psychometric(clean, mode='pooled')
plot_psychometric(result, ax=ax)
```

Analysis returns results. Plotting draws results. No exceptions.
